# ATIV-04-ET-01 – Análise do Dataset


## Upload do Dataset


In [9]:
!ls


dataset-04.zip	sample_data


## Descompactar o Dataset

In [10]:
!unzip dataset-04.zip -d dataset

Archive:  dataset-04.zip
  inflating: dataset/Image/0.jpg     
  inflating: dataset/Image/1.jpg     
  inflating: dataset/Image/10.jpg    
  inflating: dataset/Image/1000.jpg  
  inflating: dataset/Image/1001.jpg  
  inflating: dataset/Image/1002.jpg  
  inflating: dataset/Image/1003.jpg  
  inflating: dataset/Image/1004.jpg  
  inflating: dataset/Image/1005.jpg  
  inflating: dataset/Image/1006.jpg  
  inflating: dataset/Image/1007.jpg  
  inflating: dataset/Image/1008.jpg  
  inflating: dataset/Image/1009.jpg  
  inflating: dataset/Image/1010.jpg  
  inflating: dataset/Image/1011.jpg  
  inflating: dataset/Image/1012.jpg  
  inflating: dataset/Image/1013.jpg  
  inflating: dataset/Image/1014.jpg  
  inflating: dataset/Image/1015.jpg  
  inflating: dataset/Image/1016.jpg  
  inflating: dataset/Image/1017.jpg  
  inflating: dataset/Image/1018.jpg  
  inflating: dataset/Image/1019.jpg  
  inflating: dataset/Image/1020.jpg  
  inflating: dataset/Image/1021.jpg  
  inflating: dataset/Imag

In [11]:
!ls

dataset  dataset-04.zip  sample_data


## Ver o conteúdo da pasta dataset

In [12]:
!ls dataset

Image  Mask  metadata.csv


# Descrição do Dataset

O dataset utilizado nesse projeto é destinado à tarefa de segmentação de imagens. É composto por um conjunto de imagens originais `Image`, suas respectivas máscaras de segmentação `Mask` e um arquivo de metadados `metadata.csv`.

As máscaras representam as regiões de interesse a serem identificadas pelo modelo. Esta etapa tem como objetivo analisar a integridade, qualkidade e organização dos dados antes da etapa de modelagem.

## 1. Integridade dos Arquivos



### 1.1 Verificando se todas as imagens listadas no arquivo de informações existem no diretório e vice-versa

In [13]:
import os
import pandas as pd

df = pd.read_csv('dataset/metadata.csv')

img_files = set(os.listdir('dataset/Image'))
mask_files = set(os.listdir('dataset/Mask'))

img_missing = [f for f in df['Image'] if f not in img_files]
mask_missing = [f for f in df['Mask'] if f not in mask_files]

print("Linhas no metadata:", len(df))
print("Arquivos na pasta (Image):", len(img_files))
print("Arquivos na pasta (Mask):", len(mask_files))

print("Imagens do metadata que NÃO estão em Image:", len(img_missing))
print("Mascaras do metadata que NÃO estão em Mask:", len(mask_missing))

# arquivos na pasta que não estão no metadata
meta_images = set(df['Image'])
meta_masks  = set(df['Mask'])

extra_images = sorted(list(img_files - meta_images))
extra_masks  = sorted(list(mask_files - meta_masks))

print("Arquivos em Image que NÃO estão no metadata:", len(extra_images))
print("Arquivos em Mask que NÃO estão no metadata:", len(extra_masks))



Linhas no metadata: 290
Arquivos na pasta (Image): 290
Arquivos na pasta (Mask): 290
Imagens do metadata que NÃO estão em Image: 0
Mascaras do metadata que NÃO estão em Mask: 0
Arquivos em Image que NÃO estão no metadata: 0
Arquivos em Mask que NÃO estão no metadata: 0


### 1.2 Verificando o formato dos arquivos de imagem

In [14]:
#teste
import pandas as pd

df = pd.read_csv('dataset/metadata.csv')
print(df.columns)
df.head()


Index(['Image', 'Mask'], dtype='object')


,Image,Mask
0,0.jpg,0.png
1,1.jpg,1.png
2,2.jpg,2.png
3,3.jpg,3.png
4,4.jpg,4.png


In [15]:
from collections import Counter
import os

img_ext = Counter([os.path.splitext(f)[1].lower() for f in img_files])
mask_ext = Counter([os.path.splitext(f)[1].lower() for f in mask_files])

print("Extensões em Image:", img_ext)
print("Extensões em Mask:", mask_ext)


Extensões em Image: Counter({'.jpg': 290})
Extensões em Mask: Counter({'.png': 290})


### Resultado (Integridade e Formatos)

O arquivo metadata.csv contém 290 registros e foram encontrados 290 arquivos no diretório Image e 290 no diretório Mask.
Não foram identificados arquivos listados no metadata ausentes nas pastas, nem arquivos extras nas pastas que não estejam no metadata.

Quanto aos formatos, as imagens do diretório Image estão em .jpg e as máscaras do diretório Mask estão em .png,
mantendo consistência de formato dentro de cada diretório.


## 2. Consistência dos Metadados

### 2.1 Imports e Configurações

In [20]:
import os
import pandas as pd
import numpy as np
import hashlib
from PIL import Image
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Configurações de visualização
plt.style.use('ggplot')
sns.set_palette("husl")

# Caminhos
METADATA_PATH = 'dataset/metadata.csv'
IMAGE_DIR = 'dataset/Image'
MASK_DIR = 'dataset/Mask'

df = pd.read_csv(METADATA_PATH)

print(f"Total de linhas no metadata: {len(df)}")
print(f"Total de colunas: {len(df.columns)}")
print(f"\nColunas disponíveis: {list(df.columns)}")
print("\n" + "="*60)
df.head(10)

Total de linhas no metadata: 290
Total de colunas: 2

Colunas disponíveis: ['Image', 'Mask']



,Image,Mask
0,0.jpg,0.png
1,1.jpg,1.png
2,2.jpg,2.png
3,3.jpg,3.png
4,4.jpg,4.png
5,5.jpg,5.png
6,6.jpg,6.png
7,7.jpg,7.png
8,8.jpg,8.png
9,9.jpg,9.png


### 2.2 Verificação de valores ausentes, duplicatas e iconsistências nos metadados

In [21]:
print("ANÁLISE DE VALORES AUSENTES NO METADATA")
print("="*60)

# Verificar valores nulos
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Coluna': missing_values.index,
    'Valores Ausentes': missing_values.values,
    'Percentual (%)': missing_percentage.values
})

print(missing_df.to_string(index=False))

# Verificar valores vazios (strings vazias)
empty_images = df[df['Image'].astype(str).str.strip() == ''].shape[0]
empty_masks = df[df['Mask'].astype(str).str.strip() == ''].shape[0]

print(f"\nStrings vazias em 'Image': {empty_images}")
print(f"Strings vazias em 'Mask': {empty_masks}")

if missing_values.sum() == 0 and empty_images == 0 and empty_masks == 0:
    print("\nNenhum valor ausente encontrado.")
else:
    print("\nAtenção: valores ausentes detectados!")

ANÁLISE DE VALORES AUSENTES NO METADATA
Coluna  Valores Ausentes  Percentual (%)
 Image                 0             0.0
  Mask                 0             0.0

Strings vazias em 'Image': 0
Strings vazias em 'Mask': 0

Nenhum valor ausente encontrado!


In [22]:
print("ANÁLISE DE DUPLICATAS NO METADATA")
print("="*60)

# Duplicatas completas (todas as colunas)
duplicates_full = df.duplicated()
print(f"Linhas completamente duplicadas: {duplicates_full.sum()}")

if duplicates_full.sum() > 0:
    print("\nLinhas duplicadas encontradas:")
    print(df[duplicates_full])

# Duplicatas na coluna 'Image'
duplicates_image = df['Image'].duplicated()
print(f"\nImagens duplicadas (coluna 'Image'): {duplicates_image.sum()}")

if duplicates_image.sum() > 0:
    print("\nImagens que aparecem mais de uma vez:")
    dup_images = df[df['Image'].duplicated(keep=False)].sort_values('Image')
    print(dup_images)

# Duplicatas na coluna 'Mask'
duplicates_mask = df['Mask'].duplicated()
print(f"\nMáscaras duplicadas (coluna 'Mask'): {duplicates_mask.sum()}")

if duplicates_mask.sum() > 0:
    print("\nMáscaras que aparecem mais de uma vez:")
    dup_masks = df[df['Mask'].duplicated(keep=False)].sort_values('Mask')
    print(dup_masks)

if duplicates_full.sum() == 0:
    print("\nNenhuma duplicata encontrada no metadata.")

ANÁLISE DE DUPLICATAS NO METADATA
Linhas completamente duplicadas: 0

Imagens duplicadas (coluna 'Image'): 0

Máscaras duplicadas (coluna 'Mask'): 0

Nenhuma duplicata encontrada no metadata!


In [23]:
print("CONSISTÊNCIA: METADATA vs SISTEMA DE ARQUIVOS")
print("="*60)

# Listar arquivos nas pastas
img_files = set(os.listdir(IMAGE_DIR))
mask_files = set(os.listdir(MASK_DIR))

# Arquivos mencionados no metadata
meta_images = set(df['Image'])
meta_masks = set(df['Mask'])

# Arquivos faltando (no metadata mas não na pasta)
img_missing = sorted(list(meta_images - img_files))
mask_missing = sorted(list(meta_masks - mask_files))

# Arquivos extras (na pasta mas não no metadata)
extra_images = sorted(list(img_files - meta_images))
extra_masks = sorted(list(mask_files - meta_masks))

# Exibir resultados
print(f"\nArquivos na pasta 'Image': {len(img_files)}")
print(f"Arquivos na pasta 'Mask': {len(mask_files)}")
print(f"Imagens no metadata: {len(meta_images)}")
print(f"Máscaras no metadata: {len(meta_masks)}")

print("\n" + "-"*60)
print(f"Imagens do metadata que NÃO estão na pasta: {len(img_missing)}")
if img_missing:
    print(f"   Exemplos: {img_missing[:5]}")

print(f"\nMáscaras do metadata que NÃO estão na pasta: {len(mask_missing)}")
if mask_missing:
    print(f"   Exemplos: {mask_missing[:5]}")

print("\n" + "-"*60)
print(f"Arquivos na pasta 'Image' que NÃO estão no metadata: {len(extra_images)}")
if extra_images:
    print(f"   Exemplos: {extra_images[:5]}")

print(f"\nArquivos na pasta 'Mask' que NÃO estão no metadata: {len(extra_masks)}")
if extra_masks:
    print(f"   Exemplos: {extra_masks[:5]}")

if len(img_missing) == 0 and len(mask_missing) == 0 and len(extra_images) == 0 and len(extra_masks) == 0:
    print("\nMetadata e arquivos são consistentes.")

CONSISTÊNCIA: METADATA vs SISTEMA DE ARQUIVOS

Arquivos na pasta 'Image': 290
Arquivos na pasta 'Mask': 290
Imagens no metadata: 290
Máscaras no metadata: 290

------------------------------------------------------------
Imagens do metadata que NÃO estão na pasta: 0

Máscaras do metadata que NÃO estão na pasta: 0

------------------------------------------------------------
Arquivos na pasta 'Image' que NÃO estão no metadata: 0

Arquivos na pasta 'Mask' que NÃO estão no metadata: 0

Metadata e arquivos são consistentes.


Resumo:

## 3. Qualidade das Imagens

### 3.1 Verificação de imagens corrompidas e análise de dimensões

In [24]:
def check_corrupted(folder, desc="Imagens"):
    corrupted = []
    files = os.listdir(folder)

    for file in tqdm(files, desc=f"Verificando {desc}"):
        path = os.path.join(folder, file)
        try:
            with Image.open(path) as img:
                img.verify()  # Verifica integridade
        except Exception as e:
            corrupted.append({'file': file, 'error': str(e)})

    return corrupted

print("VERIFICANDO QUALIDADE DAS IMAGENS")
print("="*60)

corrupted_images = check_corrupted(IMAGE_DIR, "Imagens")
corrupted_masks = check_corrupted(MASK_DIR, "Máscaras")

print(f"\nImagens corrompidas: {len(corrupted_images)}")
if corrupted_images:
    print("   Arquivos com problema:")
    for item in corrupted_images[:10]:
        print(f"   - {item['file']}: {item['error']}")

print(f"\nMáscaras corrompidas: {len(corrupted_masks)}")
if corrupted_masks:
    print("   Arquivos com problema:")
    for item in corrupted_masks[:10]:
        print(f"   - {item['file']}: {item['error']}")

if len(corrupted_images) == 0 and len(corrupted_masks) == 0:
    print("\nTodas as imagens e máscaras estão íntegras.")

VERIFICANDO QUALIDADE DAS IMAGENS


Verificando Máscaras: 100%|██████████| 290/290 [00:00<00:00, 8227.18it/s]


Imagens corrompidas: 0

Máscaras corrompidas: 0

Todas as imagens e máscaras estão íntegras.


In [25]:
def get_image_sizes(folder, desc="Imagens"):
    sizes = []
    files = os.listdir(folder)

    for file in tqdm(files, desc=f"Analisando dimensões - {desc}"):
        path = os.path.join(folder, file)
        try:
            with Image.open(path) as img:
                sizes.append({'file': file, 'width': img.width, 'height': img.height})
        except:
            sizes.append({'file': file, 'width': None, 'height': None})

    return pd.DataFrame(sizes)

print("ANÁLISE DE DIMENSÕES")
print("="*60)

image_sizes_df = get_image_sizes(IMAGE_DIR, "Imagens")
mask_sizes_df = get_image_sizes(MASK_DIR, "Máscaras")

# Dimensões únicas
unique_image_sizes = image_sizes_df.dropna()[['width', 'height']].drop_duplicates()
unique_mask_sizes = mask_sizes_df.dropna()[['width', 'height']].drop_duplicates()

print(f"\nDimensões únicas nas imagens: {len(unique_image_sizes)}")
print(f"Dimensões únicas nas máscaras: {len(unique_mask_sizes)}")

# Estatísticas
print("\nEstatísticas das Imagens:")
print(image_sizes_df[['width', 'height']].describe())

print("\nEstatísticas das Máscaras:")
print(mask_sizes_df[['width', 'height']].describe())

ANÁLISE DE DIMENSÕES


Analisando dimensões - Máscaras: 100%|██████████| 290/290 [00:00<00:00, 16687.68it/s]


Dimensões únicas nas imagens: 201
Dimensões únicas nas máscaras: 200

Estatísticas das Imagens:
             width       height
count   290.000000   290.000000
mean   1155.179310   732.682759
std     906.546274   595.548158
min     330.000000   219.000000
25%     635.500000   414.000000
50%     900.000000   540.000000
75%    1200.000000   743.250000
max    5472.000000  3648.000000

Estatísticas das Máscaras:
             width       height
count   290.000000   290.000000
mean   1158.782759   733.696552
std     907.362182   595.808367
min     330.000000   219.000000
25%     635.500000   414.000000
50%     900.000000   540.000000
75%    1200.000000   749.000000
max    5472.000000  3648.000000
